# 📑 Student Assignment

> **Format:** Scientific report (approx. 10 pages)  
> **Hand in time:** 30.09.2026  
> **Hand in to:** Prof. Marzahn

---

### 📋 Topics that need to be addressed

* **Methodology Review:** Discussion of advantages and disadvantages of indices, radiative transfer model and crop models (**take agricultural applications into account, use your background experience**).
* **Model Calibration:** Calibration of Wofost for wheat/maize field (**What parameter values were used?**).
* **Calibration Analysis:** Discussion of calibration (**purpose, usage, etc.**).
* **Data Evaluation:** 
    * Discussion of model input data sets (**Which model input data sets were used?**).
    * Discussion of model results compared to in-situ data used for calibration purpose.
* **Synthesis:** Transferability of the calibrated Wofost model parameters

## 🌾 WOFOST Model

📘 **Reference:** [WOFOST Handbook (PDF)](https://backend.wur.nl/sites/default/files/2025-10/Gentle-WOFOST-2024.pdf)

---

### ⚙️ Modifiable Parameters

The model configuration can be adjusted across four core data domains:

* **🌤️ Weather Data:** Climate forcing inputs (radiation, temperature, rainfall, etc.).
* **🟫 Soil Data:** Physical soil properties, water retention, and rooting limits.
* **🌽 Crop Data:** Genetic crop parameters and development rates.
* **🚜 WOFOST Parameter Set:** Agromanagement settings (sowing dates, harvesting dates, and field management).

---
### 🌤️ Weather data

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os

from pcse.input import NASAPowerWeatherDataProvider

# e.g. Rostock
lat = 54.0887
lon = 12.14049

# Fetch the data
meteo_data = NASAPowerWeatherDataProvider(latitude=lat, longitude=lon)

print(meteo_data)

---
### 🟫 Soil data

In [ ]:
from pcse.input import CABOFileReader

### --- Things to Change ---
soil_name = "ec4.new"
### ---
# other soil data are stored in the folder "SF_data/Wofost_soil"
soil_path = os.path.join("SF_data/Wofost_soil", soil_name)

soil_data = CABOFileReader(soil_path)

print(soil_data)

---
### 🌽 Crop data


In [ ]:
from pcse.fileinput import YAMLCropDataProvider

# Initialize the built-in database
crop_db = YAMLCropDataProvider()

# Get the dictionary of crops and varieties
crop_dict = crop_db.get_crops_varieties()

# Define the priority crops
priority_crops = ['wheat', 'maize']

print("--- Priority Crops ---")
# Print priority crops
for crop_name in priority_crops:
    if crop_name in crop_dict:
        varieties = crop_dict[crop_name]
        print(f"\n Crop: {crop_name}")
        for variety in varieties:
            print(f"  └── {variety}")

print("\n-------------------------------------------")

# Collect all other crops and join them into a single line
other_crops = [crop.upper() for crop in crop_dict.keys() if crop not in priority_crops]

print("Other Available Crops:", ", ".join(other_crops))


In [ ]:
### --- Things to Change ---
crop = 'wheat'
crop_variety = 'Winter_wheat_101'
### ---

crop_db.set_active_crop(crop, crop_variety)

print("--- Full Active Crop Parameters ---")

for parameter, value in crop_db.items():
    if isinstance(value, list):
        print(f"🔹 {parameter}:")
        print(f"   {value}")
    else:
        print(f"🔹 {parameter}: {value}")

---
## 🚜 Run Wofost model (change parameter set)

In [ ]:
from wofost_utils import *
results = {}
wofost_parameter_sweep2(soil_path, crop, crop_variety, meteo_data);

---
## 📊 In-Situ Data (MNI Test Site - 2017)

### 🌾 Crop Types
* **Winter Wheat**
* **Maize**

### 📏 Field Measurements
* **LAI** (Leaf Area Index)
* **Dry Biomass**
* **Vegetation Height**
* **BBCH** (Crop Phenology Stage)
* **Soil Moisture**

#### 🌾 Prepare vegetation in-situ dataset

In [ ]:
### --- Things to Change ---
csv_path = "SF_data/veg_winter_wheat_508.csv"  # Points to the file in the workspace directory
### ---

print("📋 Loading VWC validation dataset...")

# Read the CSV with its semicolon separator
df_csv = pd.read_csv(csv_path, sep=';', low_memory=False)

# Clean whitespaces around header titles to avoid key errors
df_csv.columns = [str(col).strip() for col in df_csv.columns]

# Parse dates and drop rows containing invalid/missing values
df_csv['date'] = pd.to_datetime(df_csv['date'], errors='coerce')
df_csv = df_csv.dropna(subset=['date'])

# Force cast the new ground truth VWC column to standard floats
df_csv['VWC_observed'] = pd.to_numeric(df_csv['VWC kg/m2'], errors='coerce')
df_csv = df_csv.dropna(subset=['VWC_observed']).sort_values('date')


# Convert Dates to Day of Year (1 to 365)
df_csv['DOY'] = df_csv['date'].dt.dayofyear

df_csv = df_csv.rename(columns={'LAI mean': 'LAI'})

print("Original Data with DOY mapping:")
print(df_csv) 
print("\n" + "="*40 + "\n")

# Extend to a full 365-Day Time Series
full_year_doy = pd.DataFrame({'DOY': range(1, 366)})
df_extended = pd.merge(full_year_doy, df_csv, on='DOY', how='left')

# Drop the original date column since it contains NaNs for the extended days
df_extended = df_extended.drop(columns=['date'])

# Handle the days where the crop hasn't grown yet

# A. Before your first observation, LAI should be 0.0 (pre-emergence)
# We fill the first row with 0.0 so the interpolation has a starting point.
df_extended.loc[df_extended['DOY'] == 1, 'LAI'] = 0.0
df_extended.loc[df_extended['DOY'] == 1, 'Dry biomass total kg/m2'] = 0.0

# B. INTERPOLATE: Now this will safely find your real data points!
df_extended['LAI'] = df_extended['LAI'].interpolate(method='linear')
df_extended['Dry biomass total kg/m2'] = df_extended['Dry biomass total kg/m2'].interpolate(method='linear')

# C. After your last observation (post-harvest), LAI should go back to 0.0
df_extended['LAI'] = df_extended['LAI'].fillna(0.0)
df_extended['Dry biomass total kg/m2'] = df_extended['Dry biomass total kg/m2'].fillna(0.0)

print("Extended and Interpolated 365-Day Time Series:")
df_extended.set_index('DOY', inplace=True)

# Let's print a slice where your data points are filling in
print(df_extended.loc[130:145])

#### 💧 Prepare Soil Moisture In-Situ Dataset

* **Port 1:** Soil probe at **5 cm** depth
* **Port 2:** Soil probe at **5 cm** depth
* **Port 3:** Soil probe at **10 cm** depth
* **Port 4:** Soil probe at **10 cm** depth
* **Port 5:** Soil probe at **30 cm** depth

In [ ]:
# --- PRE-PROCESSING: Align In-Situ data to DOY ---
# Load data and convert 10-minute intervals to a daily mean
# Port1_SM can be change to other ports (e.g., Port2_SM, ...)
# Load the original soil moisture file

### --- Things to Change ---
df_sm = pd.read_csv("SF_data/sm_winter_wheat_508.csv", parse_dates=['date'])
port = 'Port1_SM'
### ---

df_sm.columns = df_sm.columns.str.strip()
df_sm.set_index('date', inplace=True)
sm = df_sm[port].resample('D').mean().to_frame()

# Extract DOY from the date index and make it the new index
sm['DOY'] = sm.index.dayofyear
sm.set_index('DOY', inplace=True)

# Create a complete 1 to 365 timeline
full_year_doy = pd.DataFrame({'DOY': range(1, 366)}).set_index('DOY')
sm = full_year_doy.join(sm[port], how='left')

print(sm[100:200])

---
## 📉 Compare Wofost with in-situ data

### Load Wofost run
- crop_wheat-Winter_wheat_105_soil_ec4_dates.....

In [ ]:
# Define the path to the Wofost CSV file
# Replace 'your_file.csv' with the actual name or path of your file

### --- Things to Change ---
file_path = 'your_file.csv'
# file_path = 'crop_wheat-Winter_wheat_105_soil_ec4_dates_20170301_to_20170801-span_40.0-tdwi_20.0-tsum1_750.0-tsum2_859.0-tsumem_70.0-rgrlai_0.051000000000000004-wav_5.0-cvo_0.72-cvl_0.72-LIM.csv'
### ---

try:
    # Load the CSV file into a pandas DataFrame
    wofost = pd.read_csv(file_path)
    
    print("✓ File successfully loaded!")
    print("-" * 40)
    
    # Print the first 5 rows to verify the data structure
    print("First 5 rows of the data:")
    print(wofost.head())
    
    print("-" * 40)
    # Show a summary of data types and missing values
    print("Data structure summary:")
    print(wofost.info())

except FileNotFoundError:
    print(f"Error: The file '{file_path}' was not found. Please check the file path.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

### 🌿 LAI

In [ ]:
# Create a single figure layout
fig, ax = plt.subplots(figsize=(10, 6))

# Plot WOFOST using its index as the X-axis
wofost.plot(y="LAI", ax=ax, label="WOFOST Simulation", color="teal", linewidth=2.5)

# Plot in-situ data using its index (DOY) as the X-axis
df_extended["LAI"].plot(ax=ax, label="In-Situ (Interpolated)", color="darkorange", linestyle="--")

# Customize the chart aesthetics
ax.set_title("WOFOST Model vs. In-Situ Leaf Area Index (LAI)", fontsize=14, fontweight="bold")
ax.set_xlabel("Day of Year (DOY)", fontsize=12)
ax.set_ylabel("LAI ($m^2 / m^2$)", fontsize=12)
ax.set_xlim(1, 365)
ax.grid(True, linestyle="--", alpha=0.6)
ax.legend(fontsize=11)

plt.show()

### 🍂 Dry biomass

In [ ]:
# Create a single figure layout
fig, ax = plt.subplots(figsize=(10, 6))

# Plot WOFOST using its index as the X-axis
ax.plot(wofost.index, wofost["TAGP"] / 10000, label="WOFOST Simulation", color="teal", linewidth=2.5)

# Plot in-situ data using its index (DOY) as the X-axis
df_extended["Dry biomass total kg/m2"].plot(ax=ax, label="In-Situ (Interpolated)", color="darkorange", linestyle="--")

# Customize the chart aesthetics
ax.set_title("WOFOST Model vs. In-Situ Dry biomass total kg/m2", fontsize=14, fontweight="bold")
ax.set_xlabel("Day of Year (DOY)", fontsize=12)
ax.set_ylabel("Dry biomass total kg/m2", fontsize=12)
ax.set_xlim(1, 365)
ax.grid(True, linestyle="--", alpha=0.6)
ax.legend(fontsize=11)

plt.show()

### 💧 Soil moisture

In [ ]:
# Create a single figure layout
fig, ax = plt.subplots(figsize=(10, 6))

# Plot WOFOST using its index (DOY) as the X-axis
ax.plot(wofost.index, wofost["SM"], label="WOFOST Simulation", color="teal", linewidth=2.5)

# Plot in-situ data using its index (DOY) as the X-axis
sm[port].plot(ax=ax, label="In-Situ (" + port + ")", color="darkorange", linestyle="--")

# Customize the chart aesthetics
ax.set_title("WOFOST Model vs. In-Situ Soil Moisture", fontsize=14, fontweight="bold")
ax.set_xlabel("Day of Year (DOY)", fontsize=12)
ax.set_ylabel("Volumetric Soil Moisture ($cm^3 / cm^3$)", fontsize=12)
ax.set_xlim(1, 365)
ax.grid(True, linestyle="--", alpha=0.6)
ax.legend(fontsize=11)

plt.show()